# Stage 8B-3A — Symmetric Gamma–Gamma \(N\)-Convergence

Bu aşamada hedefi kilitliyoruz:

\[
\boxed{\text{Raw empirical } q_{05} \text{ kullanmıyoruz}}
\]

Bizim dağılım varsayımımız:

\[
\boxed{\text{Symmetric Gamma–Gamma}}
\]

ve her \((x,W,z)\) için label:

\[
\boxed{
q_{05,GG}
=
GG_{0.05}\left(\mu_{\rm SNR},\sigma_{\rm emp}^{2}\right)
}
\]

olacak.

Burada:

- \(\mu_{\rm SNR}\): **analitik mean**
- \(\sigma_{\rm emp}^{2}\): CUDA Monte-Carlo ile bulunan empirical variance

Simetrik GG için:

\[
CV^2=\frac{\sigma^2}{\mu^2},
\qquad
a=b=
\frac{\sqrt{1+CV^2}+1}{CV^2}.
\]

Mevcut projenin `q05GammaGammaFit` lookup ilişkisini mevcut corrected dataset'ten
yeniden kuruyoruz.

## B3-A'nın sorusu

Yeni dataset için gerçekten kaç channel realization gerekiyor?

Tek bir \(N=100000\) stream üretip prefix'lerden:

\[
N=
1000,2000,4000,8000,10000,16000,32000,64000,100000
\]

değerlerini çıkaracağız.

Referans:

\[
\boxed{N=100000}
\]

olacak.

Karşılaştırılacak ana değerler:

\[
\sigma_{\rm emp}^2(N)
\]

ve

\[
q_{05,GG}(N).
\]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, shutil, time
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

required = [
    'ris_gpu_physics_stage1.py',
    'ris_gpu_channel_realizations_stage8b1.py',
    'ris_gpu_channel_native_stage8b2.py',
    'ris_gpu_precoder_stage5.py',
    'ris_gpu_ris_response_stage4.py',
    'stage8b2_dataset_mean_var_validation.py',
    'ris_gpu_symmetric_gg_stage8b3.py',
]

for d in [ROOT,Path('/content')]:
    if str(d) not in sys.path:
        sys.path.insert(0,str(d))

missing=[
    f for f in required
    if not (ROOT/f).exists() and not (Path('/content')/f).exists()
]
assert not missing, "Eksik:\n"+"\n".join(missing)

from ris_gpu_symmetric_gg_stage8b3 import (
    build_gg_lookup_from_dataset,
    select_convergence_rows,
    run_prefix_convergence_row,
    summarize_n_convergence,
    recommend_n,
)

device='cuda' if torch.cuda.is_available() else 'cpu'
print("Device:",device)
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))

In [ ]:
DRIVE_CSV = ROOT / 'variance_training_dataset_corrected_mu_rho_cf_v3.csv'
assert DRIVE_CSV.exists(), DRIVE_CSV

LOCAL_CSV = Path('/content/variance_training_dataset_corrected_mu_rho_cf_v3.csv')

if (
    not LOCAL_CSV.exists()
    or LOCAL_CSV.stat().st_size != DRIVE_CSV.stat().st_size
):
    print("CSV local SSD'ye kopyalanıyor...")
    shutil.copy2(DRIVE_CSV,LOCAL_CSV)

# B3 için q05GammaGammaFit gerekli, o yüzden full CSV okuyoruz.
D=pd.read_csv(LOCAL_CSV)

print("Rows:",len(D))
print("nRIS:",sorted(D.nRIS.unique()))
print("nEval unique:",sorted(D.nEval.unique())[:10])

required_cols=[
    'muSNR','varEmp','q05GammaGammaFit',
    'bankID','pairID','splitID','scenario_BR','scenario_RU',
]
assert all(c in D.columns for c in required_cols)

## 1. Mevcut Symmetric-GG lookup'ını yeniden kur

Eski pipeline'da gerçek GG target:

\[
\texttt{q05GammaGammaFit}
\]

idi.

Dataset'ten:

\[
\log CV^2
=
\log\left(\frac{\texttt{varEmp}}{\texttt{muSNR}^2}\right)
\]

ve

\[
q_{\rm norm}
=
\frac{\texttt{q05GammaGammaFit}}{\texttt{muSNR}}
\]

ilişkisini çıkarıyoruz.

In [ ]:
lookup=build_gg_lookup_from_dataset(D)

print("Lookup points:",len(lookup.log_cv2))
print("CV2 range    :",lookup.cv2_min,"->",lookup.cv2_max)
print("qNorm range  :",lookup.qnorm.min(),"->",lookup.qnorm.max())

## 2. Representative configurations

Default olarak:

\[
8\ \text{scenario branch}
\times
2\ \text{farklı bank}
=
\boxed{16\ configuration}
\]

kullanıyoruz.

Böylece \(nRIS=64,128,256,512\) hepsi test edilir.

In [ ]:
ROWS_PER_CASE=2

S=select_convergence_rows(
    D,
    split='test_interpolation',
    rows_per_case=ROWS_PER_CASE,
)

print("Selected configurations:",len(S))
display(
    S[
        [
            'bankID','pairID','scenario_BR','scenario_RU',
            'nT','nR','nRIS','nEval',
            'muSNR','varEmp','q05GammaGammaFit'
        ]
    ]
)

## 3. Tek stream, prefix convergence

Önemli: her \(N\) için kanalı yeniden üretmiyoruz.

Bir configuration için:

\[
100000
\]

realization bir kere üretiliyor.

İlk 1k, ilk 2k, ..., ilk 100k kümülatif momentlerden çıkarılıyor.

Variance convention eski dataset ile aynıdır:

\[
\boxed{
\operatorname{varEmp}
=
\operatorname{var}(Y,1)
=
E[Y^2]-E[Y]^2
}
\]

In [ ]:
N_GRID=[
    1_000,
    2_000,
    4_000,
    8_000,
    10_000,
    16_000,
    32_000,
    64_000,
    100_000,
]

all_parts=[]

if torch.cuda.is_available():
    torch.cuda.synchronize()
T0=time.perf_counter()

for i,row in S.iterrows():
    print(
        f"\n[{i+1}/{len(S)}] "
        f"bank={int(row.bankID)} pair={int(row.pairID)} | "
        f"{row.scenario_BR}/{row.scenario_RU} | "
        f"nRIS={int(row.nRIS)}"
    )

    part=run_prefix_convergence_row(
        row,
        lookup,
        n_grid=N_GRID,
        chunk_size=1000,
        device=device,
        parity=False,
    )
    all_parts.append(part)

    r10=part.loc[part.N==10_000].iloc[0]
    r100=part.loc[part.N==100_000].iloc[0]

    print(
        f"  N=10k  var err vs 100k = "
        f"{100*r10.varRelErr_vs_Nmax:.3f}% | "
        f"GG q05 err = {100*r10.q05GGRelErr_vs_Nmax:.3f}%"
    )
    print(
        f"  N=100k var={r100.varEmpPrefix:.6g} | "
        f"GG q05={r100.q05GGPrefix:.6g}"
    )
    print(
        f"  100k elapsed={r100.elapsedPrefix_s:.3f}s"
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

TOTAL_WALL=time.perf_counter()-T0

C=pd.concat(all_parts,ignore_index=True)

print("\nTotal wall:",f"{TOTAL_WALL:.3f}s = {TOTAL_WALL/60:.3f} min")

In [ ]:
summary=summarize_n_convergence(C)

display(summary)

N_RECOMMENDED=recommend_n(summary)

print("="*86)
print("STAGE 8B-3A — SYMMETRIC GG N-CONVERGENCE")
print("="*86)

if N_RECOMMENDED is None:
    print("Mevcut kriterlerle 100k altinda otomatik N seçilemedi.")
else:
    print("Önerilen minimum N:",f"{N_RECOMMENDED:,}")

print()
print("Kriterler:")
print("  GG q05 median APE <= 1%")
print("  GG q05 P90 APE    <= 2%")
print("  variance median   <= 2%")
print("  variance P90      <= 5%")

## 4. nRIS bazında kontrol

Tek global \(N\) seçmek zorunda değiliz.

Eğer örneğin:

\[
nRIS=64,128
\]

için 8k yeterli ama

\[
nRIS=512
\]

için 16k gerekiyorsa bunu burada göreceğiz.

In [ ]:
by_nris=[]

for (N,nris),g in C.groupby(['N','nRIS'],sort=True):
    by_nris.append({
        'N':int(N),
        'nRIS':int(nris),
        'configs':len(g),
        'varMdAPE_pct':100*g.varRelErr_vs_Nmax.median(),
        'varP90APE_pct':100*g.varRelErr_vs_Nmax.quantile(.90),
        'q05GGMdAPE_pct':100*g.q05GGRelErr_vs_Nmax.median(),
        'q05GGP90APE_pct':100*g.q05GGRelErr_vs_Nmax.quantile(.90),
    })

by_nris=pd.DataFrame(by_nris)
display(by_nris)

## 5. Eski 10k dataset ile bağımsız cross-check

Eski dataset:

\[
N_{\rm eval}=9997
\]

ile üretildi.

Yeni Python prefix \(N=10000\) ise **aynı random sample seti değildir**.
Bu nedenle birebir eşitlik beklenmez.

Buradaki tablo yalnızca independent-Monte-Carlo sanity check'tir.

In [ ]:
C10=C.loc[C.N==10_000].copy()

display(
    C10[
        [
            'bankID','pairID','nRIS',
            'varEmpPrefix','varEmp_dataset','varRelErr_vs_dataset',
            'q05GGPrefix','q05GG_dataset','q05GGRelErr_vs_dataset',
        ]
    ]
)

print(
    "10k Python vs old ~10k dataset | "
    "var median diff:",
    f"{100*C10.varRelErr_vs_dataset.median():.3f}%"
)
print(
    "10k Python vs old ~10k dataset | "
    "GG q05 median diff:",
    f"{100*C10.q05GGRelErr_vs_dataset.median():.3f}%"
)

In [ ]:
OUT1=Path('/content/stage8b3_symmetric_gg_prefix_results.csv')
OUT2=Path('/content/stage8b3_symmetric_gg_n_summary.csv')
OUT3=Path('/content/stage8b3_symmetric_gg_by_nris.csv')

C.to_csv(OUT1,index=False)
summary.to_csv(OUT2,index=False)
by_nris.to_csv(OUT3,index=False)

print(OUT1)
print(OUT2)
print(OUT3)

# Bundan sonra: Stage 8B-3B

B3-A sonucundan \(N\)'yi kilitledikten sonra B3-B production motorunu kuracağız.

B3-B'de aynı channel realization bankasını yeniden kullanarak:

\[
K\ \text{adet }W
\]

ve

\[
C\ \text{adet }z
\]

için empirical variance'ları batch hesaplayacağız.

Raw empirical percentile tutulmayacağı için devasa

\[
[N,K,C]
\]

\(Y\) tensörünü saklamak gerekmeyecek.

Sadece streaming:

\[
\sum Y,\qquad \sum Y^2
\]

tutacağız.

Sonra:

\[
\operatorname{varEmp}_{k,c}
=
E[Y^2]_{k,c}
-
E[Y]_{k,c}^2
\]

ve analitik mean ile:

\[
\boxed{
q_{05,GG}^{(k,c)}
=
GG_{0.05}
\left(
\mu_{\rm SNR}^{(k,c)},
\operatorname{varEmp}^{(k,c)}
\right)
}
\]

üreteceğiz.

B3-B'nin ana metriği:

\[
\boxed{\text{Symmetric-GG q05 labels/s}}
\]

olacak.